# Stage 01_graph_intervention: graph_intervention
Configuration, inputs and results remain in the corresponding stages directory.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
graph_intervention_experiment_dir = ROOT / 'experiment/research/instability_guided_bp/stages/01_graph_intervention'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

graph_intervention_data_paths = ExperimentPaths(ROOT, graph_intervention_experiment_dir / 'config/paths.json')


# Instability-guided graph intervention experiments

The causal variable used for candidate selection is short-probe instability reduction.
True logical labels are used only for evaluation. Figures are rendered inline. Aggregate tables and case/iteration records are written to the configured `stats` and `trace` paths under `experiment/data/`; this stage's README describes the output layout.

In [ ]:
import json
import sys
from collections import defaultdict
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from time import perf_counter
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr, wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5, TriangularCode
from ldpc import BpDecoder
from noises_ca import CircuitBuilder, ErrorRate

graph_intervention_config = json.loads((graph_intervention_experiment_dir / 'config/instability_guided_graph_intervention_experiments.json').read_text(encoding='utf-8'))
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# Expected relationship: long_iterations >= benchmark_iterations >= diagnostic_iterations >=
# probe_iterations >= 2.
# primary_candidate_test_iterations: chosen from candidate_test_iterations.
# candidate_test_iterations: each item is within the bounds 2 <= value <= probe_iterations.
# late_fraction: in (0, 1].
# message_zero_tolerance: non-negative.
# native_validation_tolerance: positive.
# ms_scaling_factor: finite and positive.
# stability_threshold: positive.
# primary_top_k: chosen from top_k_values.
# top_k_values: each item is positive.
# Expected relationship: number of entries in composite_weights == 3.
# sum of composite_weights: approximately 1.
# composite_weights: each item is non-negative.
# primary_max_row_weight: chosen from candidate_max_row_weights.
# candidate_screen_limit: positive.
# samples_per_outcome: positive.
# random_support_repetitions: positive.
# targeting_random_seeds: positive.
# cross_distance_levels: values exactly {1, 2, 3}.
# cross_distance_error_weights: values exactly {2, 3}.
# bootstrap_repetitions: positive.
# cv_folds: at least 2.
graph_intervention_data = graph_intervention_data_paths.run()
graph_intervention_input = graph_intervention_data_paths.input('error_weight')
OUTPUT = graph_intervention_data_paths.run()

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(graph_intervention_data_paths, graph_intervention_config)

STATS = graph_intervention_data_paths.output('stats')
graph_intervention_trace_dir = graph_intervention_data_paths.output('trace')
STATS.mkdir(parents=True, exist_ok=True)
graph_intervention_trace_dir.mkdir(parents=True, exist_ok=True)

from experiment.utils import misc

def log(level, payload):
    return misc.log(level, payload, CONFIG=graph_intervention_config)

sns.set_theme(style='whitegrid')
log(2, {'stats': str(STATS), 'trace': str(graph_intervention_trace_dir)})


## Instrumented BP and graph contexts

Prepare native BP traces and contexts for the candidate graph changes. A redundant check requires its corresponding syndrome entry; the intervention changes representation while retaining the decoding problem.


In [ ]:
from experiment.utils import graph_intervention

graph_intervention_GraphContext = graph_intervention.GraphContext

from experiment.utils import bp_message_dynamics

graph_intervention_MessageTrace = bp_message_dynamics.MessageTrace

@dataclass(frozen=True)
class InstabilityScores:
    trajectory: np.ndarray
    peak: np.ndarray
    sign_flip: np.ndarray
    composite: np.ndarray
    per_iteration: np.ndarray

def graph_intervention_graph_context(
    p,
    H,
    L,
    priors,
):
    return graph_intervention.graph_context(p, H, L, priors, GraphContext=graph_intervention_GraphContext)

graph_intervention_context_cache = {}

def graph_intervention_context_for(level, p):
    return graph_intervention.context_for(
        level,
        p,
        CA_9_2_5=CA_9_2_5,
        CONTEXT_CACHE=graph_intervention_context_cache,
        CircuitBuilder=CircuitBuilder,
        ErrorRate=ErrorRate,
        TriangularCode=TriangularCode,
        detector_error_model_to_check_matrices=detector_error_model_to_check_matrices,
        graph_context=graph_intervention_graph_context,
    )

def graph_intervention_augmented_context(context, extra_rows):
    return graph_intervention.augmented_context(context, extra_rows, graph_context=graph_intervention_graph_context)

from experiment.utils import misc

graph_intervention_bit_array = misc.bit_array_variant_2
graph_intervention_bit_text = misc.bit_text_variant_2

def graph_intervention_logical_label(context, correction):
    return graph_intervention.logical_label(context, correction, bit_text=graph_intervention_bit_text)

def graph_intervention_trace_min_sum(
    context,
    syndrome,
    iterations,
    edge_damping=None,
    edge_scaling=None,
):
    return graph_intervention.trace_min_sum(context, syndrome, iterations, edge_damping, edge_scaling, CONFIG=graph_intervention_config, MessageTrace=graph_intervention_MessageTrace)


In [ ]:
from experiment.utils import graph_intervention

graph_intervention_percentile_rank = graph_intervention.percentile_rank

def transient_instability(context, trace, probe_iterations):
    assert 2 <= probe_iterations < len(trace.c2v)
    messages = trace.c2v[1:probe_iterations + 1]
    delta = np.abs(np.diff(trace.c2v[:probe_iterations + 1], axis=0))
    scale = np.abs(context.prior_llr[context.edge_variables]) + np.finfo(float).eps
    normalized = delta / scale
    signs = np.where(np.abs(messages) <= graph_intervention_config['message_zero_tolerance'], 0, np.where(messages > 0, 1, -1))
    comparable = (signs[1:] != 0) & (signs[:-1] != 0)
    flips = ((signs[1:] != signs[:-1]) & comparable).astype(float)
    trajectory = np.zeros(context.H.shape[1])
    peak = np.zeros_like(trajectory)
    sign_flip = np.zeros_like(trajectory)
    per_iteration = np.zeros((probe_iterations, context.H.shape[1]))

    for variable, edge_ids in enumerate(context.variable_edges):
        if not len(edge_ids):
            continue
        per_iteration[:, variable] = normalized[:, edge_ids].mean(axis=1)
        trajectory[variable] = per_iteration[:, variable].mean()
        peak[variable] = per_iteration[:, variable].max()
        sign_flip[variable] = flips[:, edge_ids].mean() if len(flips) else 0.0

    weights = np.asarray(graph_intervention_config['composite_weights'])
    composite = weights[0] * graph_intervention_percentile_rank(trajectory) + weights[1] * graph_intervention_percentile_rank(peak) + weights[2] * graph_intervention_percentile_rank(sign_flip)

    return InstabilityScores(trajectory, peak, sign_flip, composite, per_iteration)

def normalized_late_residual(context, trace, endpoint):
    assert 2 <= endpoint < len(trace.c2v)
    delta = np.abs(np.diff(trace.c2v[:endpoint + 1], axis=0))
    late_count = max(1, int(np.ceil(graph_intervention_config['late_fraction'] * len(delta))))
    scale = np.abs(context.prior_llr[context.edge_variables]) + np.finfo(float).eps

    return float(np.max(delta[-late_count:] / scale))

def benchmark_result(
    context,
    syndrome,
    truth,
    error,
    trace,
):
    first = trace.first_valid_iteration
    index = first if first is not None and first <= graph_intervention_config['benchmark_iterations'] else graph_intervention_config['benchmark_iterations']
    hard = trace.hard[index]
    valid = bool(trace.syndrome_consistent[index])
    correct = graph_intervention_logical_label(context, hard) == truth
    late = normalized_late_residual(context, trace, graph_intervention_config['diagnostic_iterations'])
    stable = late < graph_intervention_config['stability_threshold']
    later = np.flatnonzero(trace.syndrome_consistent[index + 1:] & np.asarray([graph_intervention_logical_label(context, candidate) == truth for candidate in trace.hard[index + 1:]]))

    if valid and correct:
        failure_type = 'not_failure'
    elif not valid:
        failure_type = 'dynamic_syndrome_failure'
    elif later.size:
        failure_type = 'premature_first_valid_failure'
    elif stable:
        failure_type = 'stable_persistent_logical_failure'
    else:
        failure_type = 'unstable_persistent_logical_failure'

    residual_weight = np.nan
    residual_text = ''
    if valid:
        residual = error ^ hard
        assert not np.any(context.H @ residual % 2)
        residual_weight = int(residual.sum())
        residual_text = graph_intervention_bit_text(residual)

    return {
        'benchmark_iteration': index,
        'benchmark_syndrome_consistent': int(valid),
        'benchmark_logical_correct': int(correct),
        'benchmark_sector': graph_intervention_logical_label(context, hard),
        'max_normalized_late_residual': late,
        'stable': int(stable),
        'failure_type': failure_type,
        'residual_weight': residual_weight,
        'residual': residual_text,
    }

def graph_intervention_outcome_of(
    context,
    syndrome,
    truth,
    correction,
):
    return graph_intervention.outcome_of(context, syndrome, truth, correction, logical_label=graph_intervention_logical_label)

graph_intervention_top_region = graph_intervention.top_region

def local_instability(scores, region):
    return float(np.asarray(scores.composite)[np.asarray(region, dtype=int)].mean())

graph_intervention_conditional_outcome_rates = graph_intervention.conditional_outcome_rates

def graph_intervention_edge_maps(
    context,
    variable_scores,
    low,
    high,
    mode,
    rng=None,
):
    return graph_intervention.edge_maps(context, variable_scores, low, high, mode, rng, percentile_rank=graph_intervention_percentile_rank)


In [ ]:
from experiment.utils import graph_intervention

graph_intervention_CandidateRow = graph_intervention.CandidateRow

def graph_intervention_generate_redundant_candidates(H, max_row_weight):
    return graph_intervention.generate_redundant_candidates(H, max_row_weight, CandidateRow=graph_intervention_CandidateRow, bit_text=graph_intervention_bit_text)

graph_intervention_augmented_syndrome = graph_intervention.augmented_syndrome
graph_intervention_candidate_overlap = graph_intervention.candidate_overlap

def candidate_probe(
    context,
    syndrome,
    candidate,
    region,
    before_by_window,
):
    augmented = graph_intervention_augmented_context(context, candidate.row)
    augmented_s = graph_intervention_augmented_syndrome(syndrome, candidate.source_rows)
    rows = []

    for test_iterations in graph_intervention_config['candidate_test_iterations']:
        trace = graph_intervention_trace_min_sum(augmented, augmented_s, test_iterations)
        scores = transient_instability(augmented, trace, test_iterations)
        after = local_instability(scores, region)
        before = before_by_window[test_iterations]
        rows.append({
            'test_iterations': test_iterations,
            'I_before': before,
            'I_after': after,
            'instability_gain': before - after,
            'relative_instability_gain': (before - after) / (before + np.finfo(float).eps),
            'short_probe_syndrome_consistent': int(trace.syndrome_consistent[-1]),
        })

    return rows

def full_decode_candidate(
    context,
    syndrome,
    truth,
    candidate,
    region,
):
    augmented = graph_intervention_augmented_context(context, candidate.row)
    augmented_s = graph_intervention_augmented_syndrome(syndrome, candidate.source_rows)
    started = perf_counter()
    trace = graph_intervention_trace_min_sum(augmented, augmented_s, graph_intervention_config['benchmark_iterations'])
    runtime = perf_counter() - started
    first = trace.first_valid_iteration
    index = first if first is not None else graph_intervention_config['benchmark_iterations']
    correction = trace.hard[index]
    augmented_valid = bool(trace.syndrome_consistent[index])
    base_valid, correct = graph_intervention_outcome_of(context, syndrome, truth, correction)
    assert augmented_valid <= base_valid
    final_scores = transient_instability(augmented, trace, graph_intervention_config['benchmark_iterations'])

    return {
        'full_syndrome_consistent': int(base_valid),
        'full_logical_correct': int(correct),
        'full_augmented_syndrome_consistent': int(augmented_valid),
        'iterations': index,
        'runtime_seconds': runtime,
        'final_instability': local_instability(final_scores, region),
        'final_normalized_late_residual': normalized_late_residual(augmented, trace, graph_intervention_config['benchmark_iterations']),
        'actual_edge_updates': int(index * augmented.H.sum()),
        'configured_edge_updates': int(graph_intervention_config['benchmark_iterations'] * augmented.H.sum()),
    }

def correction_cost(context, correction):
    correction = np.asarray(correction, dtype=np.uint8)
    return float(np.where(correction, -np.log(context.priors), -np.log1p(-context.priors)).sum())


## Population and Experiment 1: transient instability maps

Construct the diagnostic population and locate transient message instability. These maps supply candidate regions; they are not assumed to be complete logical-error predictors.


In [ ]:
graph_intervention_dtype = {
    'p': float,
    'err': str,
    'det': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight2 = pd.read_csv(graph_intervention_data_paths.input('weight_2_data_csv'), dtype=graph_intervention_dtype)
weight3 = pd.read_csv(graph_intervention_data_paths.input('weight_3_data_csv'), dtype=graph_intervention_dtype)

frames = []
for source, graph_intervention_weight in ((weight2, 2), (weight3, 3)):
    assert set(graph_intervention_dtype) <= set(source.columns) and source.notna().all().all()
    frame = source.drop_duplicates(['p', 'det', 'true']).copy()
    frame['error_weight'] = graph_intervention_weight
    frame['syndrome_weight'] = frame.det.str.count('1').astype(int)
    frames.append(frame)

population = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=graph_intervention_config['random_seed']).reset_index(drop=True)
population_rows = []
variable_rows = []

for graph_intervention_sample_id, graph_intervention_row in enumerate(tqdm(population.itertuples(index=False), total=len(population), desc='Transient population', disable=graph_intervention_config['print_level'] == 1)):
    graph_intervention_, graph_intervention_context = graph_intervention_context_for(2, graph_intervention_row.p)
    graph_intervention_syndrome = graph_intervention_bit_array(graph_intervention_row.det)
    graph_intervention_error = graph_intervention_bit_array(graph_intervention_row.err)
    graph_intervention_trace = graph_intervention_trace_min_sum(graph_intervention_context, graph_intervention_syndrome, graph_intervention_config['long_iterations'])
    result = benchmark_result(graph_intervention_context, graph_intervention_syndrome, graph_intervention_row.true, graph_intervention_error, graph_intervention_trace)
    scores = transient_instability(graph_intervention_context, graph_intervention_trace, graph_intervention_config['probe_iterations'])

    population_rows.append({
        'sample_id': graph_intervention_sample_id,
        'p': graph_intervention_row.p,
        'error_weight': graph_intervention_row.error_weight,
        'syndrome_weight': graph_intervention_row.syndrome_weight,
        'error': graph_intervention_row.err,
        'syndrome': graph_intervention_row.det,
        'true_sector': graph_intervention_row.true,
        **result,
        'max_trajectory_instability': float(scores.trajectory.max()),
        'mean_trajectory_instability': float(scores.trajectory.mean()),
        'max_peak_instability': float(scores.peak.max()),
        'mean_sign_flip_instability': float(scores.sign_flip.mean()),
        'max_composite_instability': float(scores.composite.max()),
        **{f'top_{K}_variables': json.dumps(graph_intervention_top_region(scores.composite, K).tolist()) for K in graph_intervention_config['top_k_values']},
    })

    for graph_intervention_variable in range(graph_intervention_context.H.shape[1]):
        variable_rows.append({
            'sample_id': graph_intervention_sample_id,
            'failure_type': result['failure_type'],
            'variable_id': graph_intervention_variable,
            'trajectory_instability': scores.trajectory[graph_intervention_variable],
            'peak_instability': scores.peak[graph_intervention_variable],
            'sign_flip_instability': scores.sign_flip[graph_intervention_variable],
            'composite_instability': scores.composite[graph_intervention_variable],
        })

graph_intervention_population_cases = pd.DataFrame(population_rows)
variable_instability = pd.DataFrame(variable_rows)
assert len(graph_intervention_population_cases) == len(population) == 23178 and graph_intervention_population_cases.sample_id.is_unique
assert len(variable_instability) == len(graph_intervention_population_cases) * graph_intervention_context.H.shape[1]

graph_intervention_population_cases.to_csv(graph_intervention_trace_dir / 'graph_intervention_population_cases.csv', index=False)
variable_instability.to_csv(graph_intervention_trace_dir / 'transient_variable_instability.csv', index=False)
transient_summary = graph_intervention_population_cases.groupby('failure_type', as_index=False).agg(
    cases=('sample_id', 'size'),
    max_I_mean=('max_trajectory_instability', 'mean'),
    max_I_median=('max_trajectory_instability', 'median'),
    mean_I_mean=('mean_trajectory_instability', 'mean'),
    mean_I_median=('mean_trajectory_instability', 'median'),
    peak_I_mean=('max_peak_instability', 'mean'),
    flip_I_mean=('mean_sign_flip_instability', 'mean'),
)

transient_summary.to_csv(graph_intervention_data_paths.output('transient_instability_summary_csv'), index=False)
sampled_arms = []
for arm_index, (graph_intervention_failure_type, graph_intervention_group) in enumerate(graph_intervention_population_cases.groupby('failure_type', sort=True)):
    graph_intervention_count = min(graph_intervention_config['samples_per_outcome'], len(graph_intervention_group))
    sampled_arms.append(graph_intervention_group.sample(graph_intervention_count, random_state=graph_intervention_config['random_seed'] + arm_index))

intervention_cases = pd.concat(sampled_arms, ignore_index=True)
selection_audit = intervention_cases.groupby('failure_type', as_index=False).size().rename(columns={'size': 'selected_cases'})
selection_audit['available_cases'] = selection_audit.failure_type.map(graph_intervention_population_cases.failure_type.value_counts())

selection_audit.to_csv(graph_intervention_data_paths.output('graph_intervention_selection_audit_csv'), index=False)
assert intervention_cases.sample_id.is_unique
iteration_rows = []

for graph_intervention_selected in tqdm(intervention_cases.itertuples(index=False), total=len(intervention_cases), desc='Selected transient trajectories', disable=graph_intervention_config['print_level'] == 1):
    graph_intervention_, graph_intervention_context = graph_intervention_context_for(2, graph_intervention_selected.p)
    graph_intervention_trace = graph_intervention_trace_min_sum(graph_intervention_context, graph_intervention_bit_array(graph_intervention_selected.syndrome), graph_intervention_config['probe_iterations'])
    scores = transient_instability(graph_intervention_context, graph_intervention_trace, graph_intervention_config['probe_iterations'])

    for graph_intervention_iteration in range(graph_intervention_config['probe_iterations']):
        for graph_intervention_variable in range(graph_intervention_context.H.shape[1]):
            iteration_rows.append({
                'sample_id': graph_intervention_selected.sample_id,
                'failure_type': graph_intervention_selected.failure_type,
                'iteration': graph_intervention_iteration + 1,
                'variable_id': graph_intervention_variable,
                'normalized_message_residual': scores.per_iteration[graph_intervention_iteration, graph_intervention_variable],
            })

iteration_instability = pd.DataFrame(iteration_rows)
iteration_instability.to_csv(graph_intervention_trace_dir / 'transient_instability_iterations.csv', index=False)
transient_summary


## Experiment 2: instability--residual spatial overlap

Compare the unstable region with the residual error/correction structure. This is an offline spatial association test because the true error support is unavailable to a runtime policy.


In [ ]:
def variable_neighborhood(context, support, hops):
    variables = set((int(value) for value in support))
    frontier = set(variables)
    for _ in range(hops):
        checks = {int(context.edge_checks[edge]) for variable in frontier for edge in context.variable_edges[variable]}
        expanded = {int(context.edge_variables[edge]) for check in checks for edge in context.check_edges[check]}
        frontier = expanded - variables
        variables |= expanded

    return np.asarray(sorted(variables), dtype=int)

def instability_mass_fraction(instability, support):
    return float(instability[support].sum() / (instability.sum() + np.finfo(float).eps))

stable_valid = graph_intervention_population_cases[graph_intervention_population_cases.benchmark_syndrome_consistent.eq(1) & graph_intervention_population_cases.stable.eq(1)].copy()
stable_valid['stable_outcome'] = np.where(stable_valid.benchmark_logical_correct.eq(1), 'stable-valid-correct', 'stable-valid-logical-wrong')

def matched_stable_cases(frame, limit):
    correct = frame[frame.stable_outcome == 'stable-valid-correct'].sample(frac=1, random_state=graph_intervention_config['random_seed'])
    wrong = frame[frame.stable_outcome == 'stable-valid-logical-wrong'].sample(frac=1, random_state=graph_intervention_config['random_seed'] + 1)
    pairs = []
    keys = ['p', 'error_weight', 'syndrome_weight', 'residual_weight']

    for values, wrong_group in wrong.groupby(keys, sort=True):
        values = values if isinstance(values, tuple) else (values,)
        mask = np.ones(len(correct), dtype=bool)
        for key, value in zip(keys, values):
            mask &= correct[key].to_numpy() == value
        correct_group = correct.loc[mask]

        for left, right in zip(wrong_group.itertuples(index=False), correct_group.itertuples(index=False)):
            pair_id = f'overlap-{len(pairs) // 2}'
            pairs.extend([{'overlap_pair_id': pair_id, **left._asdict()}, {'overlap_pair_id': pair_id, **right._asdict()}])

    matched = pd.DataFrame(pairs)
    ids = pd.Series(matched.overlap_pair_id.unique()).sample(min(limit, matched.overlap_pair_id.nunique()), random_state=graph_intervention_config['random_seed'] + 2)
    result = matched[matched.overlap_pair_id.isin(ids)].copy()
    assert result.groupby('overlap_pair_id').size().eq(2).all()

    return result

overlap_input = matched_stable_cases(stable_valid, graph_intervention_config['samples_per_outcome'])
overlap_rows = []
for graph_intervention_row in tqdm(overlap_input.itertuples(index=False), total=len(overlap_input), desc='Residual overlap', disable=graph_intervention_config['print_level'] == 1):
    graph_intervention_, graph_intervention_context = graph_intervention_context_for(2, graph_intervention_row.p)
    sample_variables = variable_instability[variable_instability.sample_id == graph_intervention_row.sample_id]
    instability = sample_variables.sort_values('variable_id').composite_instability.to_numpy()
    residual_support = np.flatnonzero(graph_intervention_bit_array(graph_intervention_row.residual))
    graph_intervention_rng = np.random.default_rng(graph_intervention_config['random_seed'] + int(graph_intervention_row.sample_id))
    random_supports = [np.sort(graph_intervention_rng.choice(graph_intervention_context.H.shape[1], len(residual_support), replace=False)) for _ in range(graph_intervention_config['random_support_repetitions'])]

    for hops in (0, 1, 2):
        actual_region = variable_neighborhood(graph_intervention_context, residual_support, hops)
        actual = instability_mass_fraction(instability, actual_region)
        controls = np.asarray([instability_mass_fraction(instability, variable_neighborhood(graph_intervention_context, support, hops)) for support in random_supports])
        overlap_rows.append({
            'sample_id': graph_intervention_row.sample_id,
            'overlap_pair_id': graph_intervention_row.overlap_pair_id,
            'outcome_class': graph_intervention_row.stable_outcome,
            'p': graph_intervention_row.p,
            'error_weight': graph_intervention_row.error_weight,
            'syndrome_weight': graph_intervention_row.syndrome_weight,
            'residual_weight': graph_intervention_row.residual_weight,
            'hops': hops,
            'instability_residual_overlap': actual,
            'random_overlap_mean': controls.mean(),
            'random_overlap_std': controls.std(ddof=1),
            'instability_residual_enrichment': actual - controls.mean(),
        })

overlap_cases = pd.DataFrame(overlap_rows)
overlap_cases.to_csv(graph_intervention_trace_dir / 'instability_residual_overlap_cases.csv', index=False)
overlap_summary = overlap_cases.groupby(['outcome_class', 'hops'], as_index=False).agg(
    cases=('sample_id', 'size'),
    overlap_mean=('instability_residual_overlap', 'mean'),
    enrichment_mean=('instability_residual_enrichment', 'mean'),
    enrichment_median=('instability_residual_enrichment', 'median'),
)

overlap_summary.to_csv(graph_intervention_data_paths.output('instability_residual_overlap_csv'), index=False)
overlap_summary


## Experiment 3: redundant-row short-probe gain and full outcome

Probe candidate redundant rows briefly, measure the instability response and then record the full decoding outcome. A smoother probe is useful only if it predicts a meaningful repair.


In [ ]:
def balanced_screen(
    candidates,
    overlap_scores,
    limit,
    rng,
):
    if len(candidates) <= limit:
        return candidates
    order = np.lexsort((np.arange(len(candidates)), -np.asarray(overlap_scores)))
    top_count = limit // 3
    bottom_count = limit // 3
    chosen = list(order[:top_count]) + list(order[-bottom_count:])
    remaining = np.setdiff1d(np.arange(len(candidates)), chosen)
    chosen.extend(rng.choice(remaining, size=limit - len(chosen), replace=False).tolist())
    assert len(set(chosen)) == limit

    return [candidates[index] for index in chosen]

candidate_pool_rows = []
reference_context = graph_intervention_context_for(2, float(graph_intervention_population_cases.p.iloc[0]))[1]
for max_weight in graph_intervention_config['candidate_max_row_weights']:
    graph_intervention_pool = graph_intervention_generate_redundant_candidates(reference_context.H, max_weight)
    candidate_pool_rows.append({
        'max_row_weight': max_weight,
        'candidate_count': len(graph_intervention_pool),
        'minimum_weight': min((row.row_weight for row in graph_intervention_pool), default=np.nan),
        'maximum_weight': max((row.row_weight for row in graph_intervention_pool), default=np.nan),
    })

pd.DataFrame(candidate_pool_rows).to_csv(graph_intervention_data_paths.output('candidate_pool_summary_csv'), index=False)
candidate_rows = []
for graph_intervention_sample in tqdm(intervention_cases.itertuples(index=False), total=len(intervention_cases), desc='Candidate graph probes', disable=graph_intervention_config['print_level'] == 1):
    graph_intervention_, graph_intervention_context = graph_intervention_context_for(2, graph_intervention_sample.p)
    graph_intervention_syndrome = graph_intervention_bit_array(graph_intervention_sample.syndrome)
    graph_intervention_baseline_trace = graph_intervention_trace_min_sum(graph_intervention_context, graph_intervention_syndrome, graph_intervention_config['benchmark_iterations'])
    baseline_scores = transient_instability(graph_intervention_context, graph_intervention_baseline_trace, graph_intervention_config['probe_iterations'])
    graph_intervention_region = graph_intervention_top_region(baseline_scores.composite, graph_intervention_config['primary_top_k'])
    before_by_window = {value: local_instability(transient_instability(graph_intervention_context, graph_intervention_baseline_trace, value), graph_intervention_region) for value in graph_intervention_config['candidate_test_iterations']}
    all_candidates = graph_intervention_generate_redundant_candidates(graph_intervention_context.H, graph_intervention_config['primary_max_row_weight'])
    assert all_candidates
    weighted = [graph_intervention_candidate_overlap(candidate, baseline_scores.composite, graph_intervention_region)[1] for candidate in all_candidates]
    graph_intervention_rng = np.random.default_rng(graph_intervention_config['random_seed'] + int(graph_intervention_sample.sample_id))
    graph_intervention_screened = balanced_screen(all_candidates, weighted, graph_intervention_config['candidate_screen_limit'], graph_intervention_rng)

    for graph_intervention_candidate in graph_intervention_screened:
        graph_intervention_overlap_count, overlap_weighted = graph_intervention_candidate_overlap(graph_intervention_candidate, baseline_scores.composite, graph_intervention_region)
        graph_intervention_probes = candidate_probe(graph_intervention_context, graph_intervention_syndrome, graph_intervention_candidate, graph_intervention_region, before_by_window)
        graph_intervention_full = full_decode_candidate(graph_intervention_context, graph_intervention_syndrome, graph_intervention_sample.true_sector, graph_intervention_candidate, graph_intervention_region)
        new_success = bool(graph_intervention_full['full_syndrome_consistent'] and graph_intervention_full['full_logical_correct'])
        graph_intervention_baseline_success = graph_intervention_sample.failure_type == 'not_failure'

        for graph_intervention_probe in graph_intervention_probes:
            candidate_rows.append({
                'sample_id': graph_intervention_sample.sample_id,
                'failure_type': graph_intervention_sample.failure_type,
                'candidate_id': graph_intervention_candidate.candidate_id,
                'source_rows': json.dumps(graph_intervention_candidate.source_rows),
                'row_weight': graph_intervention_candidate.row_weight,
                'row_support': json.dumps(np.flatnonzero(graph_intervention_candidate.row).tolist()),
                'num_original_rows_combined': 2,
                'overlap_with_U': graph_intervention_overlap_count,
                'weighted_instability_overlap': overlap_weighted,
                'region': json.dumps(graph_intervention_region.tolist()),
                **graph_intervention_probe,
                **graph_intervention_full,
                'repair': int(not graph_intervention_baseline_success and new_success),
                'regression': int(graph_intervention_baseline_success and (not new_success)),
            })

candidate_trials = pd.DataFrame(candidate_rows)
screened_per_sample = candidate_trials.groupby('sample_id').candidate_id.nunique()
assert screened_per_sample.nunique() == 1
expected = int(screened_per_sample.iloc[0]) * len(intervention_cases) * len(graph_intervention_config['candidate_test_iterations'])
assert len(candidate_trials) == expected
assert candidate_trials.groupby(['sample_id', 'candidate_id', 'test_iterations']).size().eq(1).all()

candidate_trials.to_csv(graph_intervention_trace_dir / 'candidate_row_trials.csv', index=False)
candidate_gain_summary = candidate_trials.groupby(['test_iterations', 'failure_type'], as_index=False).agg(
    candidates=('candidate_id', 'size'),
    mean_gain=('instability_gain', 'mean'),
    median_gain=('instability_gain', 'median'),
    positive_gain_fraction=('instability_gain', lambda x: np.mean(x > 0)),
    repair_fraction=('repair', 'mean'),
    regression_fraction=('regression', 'mean'),
)
candidate_gain_summary.to_csv(graph_intervention_data_paths.output('candidate_row_gain_summary_csv'), index=False)
primary_trials = candidate_trials[candidate_trials.test_iterations == graph_intervention_config['primary_candidate_test_iterations']].copy()
gain_bins = pd.qcut(primary_trials.instability_gain.rank(method='first'), 5, labels=['0-20', '20-40', '40-60', '60-80', '80-100'])
primary_trials['gain_percentile_bin'] = gain_bins.astype(str)
repair_vs_gain = primary_trials.groupby('gain_percentile_bin', observed=True, as_index=False).agg(
    candidates=('candidate_id', 'size'),
    gain_mean=('instability_gain', 'mean'),
    repair_fraction=('repair', 'mean'),
    regression_fraction=('regression', 'mean'),
    final_instability_mean=('final_instability', 'mean'),
)
prediction = primary_trials[primary_trials.failure_type != 'not_failure']

if prediction.repair.nunique() == 2:
    repair_vs_gain['candidate_gain_auroc'] = roc_auc_score(prediction.repair, prediction.instability_gain)
else:
    repair_vs_gain['candidate_gain_auroc'] = np.nan
correlation, correlation_p = spearmanr(primary_trials.instability_gain, primary_trials.I_before - primary_trials.final_instability)
repair_vs_gain['gain_final_reduction_spearman'] = correlation
repair_vs_gain['gain_final_reduction_p_value'] = correlation_p

repair_vs_gain.to_csv(graph_intervention_data_paths.output('repair_vs_instability_gain_csv'), index=False)
candidate_gain_summary


## Experiment 4: equal-probe targeting ablation

Compare overlap targeting and response-based targeting with random and weight-matched random choices. Matching probe effort and row size tests whether the targeting information contributes beyond candidate budget.


In [ ]:
def choose_strategy_rows(group, seed):
    reduction = group.sort_values(['instability_gain', 'candidate_id'], ascending=[False, True]).iloc[0]
    overlap = group.sort_values(['weighted_instability_overlap', 'candidate_id'], ascending=[False, True]).iloc[0]
    anti = group.sort_values(['weighted_instability_overlap', 'candidate_id'], ascending=[True, True]).iloc[0]
    rng = np.random.default_rng(seed)
    random = group.iloc[int(rng.integers(len(group)))]
    matched = group[group.row_weight == reduction.row_weight]
    weight_random = matched.iloc[int(rng.integers(len(matched)))]

    return {
        'fully_random': random,
        'weight_matched_random': weight_random,
        'overlap_targeted': overlap,
        'anti_targeted': anti,
        'instability_reduction_targeted': reduction,
    }

targeting_rows = []
probe_budget_by_sample = primary_trials.groupby('sample_id').apply(lambda group: int(np.sum(group.test_iterations * (reference_context.H.sum() + group.row_weight.to_numpy()))), include_groups=False)
for graph_intervention_sample_id, graph_intervention_group in tqdm(primary_trials.groupby('sample_id'), total=primary_trials.sample_id.nunique(), desc='Targeting ablation', disable=graph_intervention_config['print_level'] == 1):
    graph_intervention_sample = intervention_cases[intervention_cases.sample_id == graph_intervention_sample_id].iloc[0]
    for graph_intervention_seed_index in range(graph_intervention_config['targeting_random_seeds']):
        graph_intervention_choices = choose_strategy_rows(graph_intervention_group, graph_intervention_config['random_seed'] + int(graph_intervention_sample_id) * 1009 + graph_intervention_seed_index)
        for graph_intervention_strategy, graph_intervention_selected in graph_intervention_choices.items():
            graph_intervention_baseline_success = graph_intervention_sample.failure_type == 'not_failure'
            new_success = bool(graph_intervention_selected.full_syndrome_consistent and graph_intervention_selected.full_logical_correct)
            targeting_rows.append({
                'sample_id': graph_intervention_sample_id,
                'failure_type': graph_intervention_sample.failure_type,
                'strategy': graph_intervention_strategy,
                'random_seed_index': graph_intervention_seed_index,
                'candidate_id': graph_intervention_selected.candidate_id,
                'row_weight': graph_intervention_selected.row_weight,
                'instability_gain': graph_intervention_selected.instability_gain,
                'final_instability_reduction': graph_intervention_selected.I_before - graph_intervention_selected.final_instability,
                'syndrome_consistent': graph_intervention_selected.full_syndrome_consistent,
                'logical_correct': graph_intervention_selected.full_logical_correct,
                'repair': int(not graph_intervention_baseline_success and new_success),
                'regression': int(graph_intervention_baseline_success and (not new_success)),
                'full_decode_edge_updates': graph_intervention_selected.actual_edge_updates,
                'candidate_probe_edge_updates': probe_budget_by_sample.loc[graph_intervention_sample_id],
                'total_edge_updates': graph_intervention_selected.actual_edge_updates + probe_budget_by_sample.loc[graph_intervention_sample_id],
                'runtime_seconds': graph_intervention_selected.runtime_seconds,
            })

targeting_cases = pd.DataFrame(targeting_rows)
assert targeting_cases.groupby(['sample_id', 'random_seed_index', 'strategy']).size().eq(1).all()
targeting_cases.to_csv(graph_intervention_trace_dir / 'targeting_ablation_cases.csv', index=False)
targeting_ablation = graph_intervention_conditional_outcome_rates(targeting_cases.groupby('strategy', as_index=False).agg(
    evaluations=('sample_id', 'size'),
    unique_cases=('sample_id', 'nunique'),
    repair_count=('repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('failure_type', lambda x: np.sum(x != 'not_failure')),
    baseline_success_evaluations=('failure_type', lambda x: np.sum(x == 'not_failure')),
    syndrome_consistent_rate=('syndrome_consistent', 'mean'),
    logical_correct_rate=('logical_correct', 'mean'),
    mean_instability_gain=('instability_gain', 'mean'),
    mean_final_instability_reduction=('final_instability_reduction', 'mean'),
    mean_edge_updates=('total_edge_updates', 'mean'),
    median_runtime_seconds=('runtime_seconds', 'median'),
))
targeting_ablation['net_gain'] = [int(group.repair.sum() - group.regression.sum()) for _, group in targeting_cases.groupby('strategy', sort=True)]

targeting_ablation.to_csv(graph_intervention_data_paths.output('targeting_ablation_csv'), index=False)
targeting_ablation


## Experiments 5--6: local damping and min-sum scaling

Compare local and global damping/scaling with graph interventions. These controls test whether simple numerical stabilization explains the observed benefit.


In [ ]:
def evaluate_modified(
    context,
    base_context,
    syndrome,
    augmented_syndrome_value,
    truth,
    failure_type,
    baseline_I,
    region,
    damping=None,
    scaling=None,
):
    started = perf_counter()
    trace = graph_intervention_trace_min_sum(context, augmented_syndrome_value, graph_intervention_config['benchmark_iterations'], damping, scaling)
    runtime = perf_counter() - started
    first = trace.first_valid_iteration
    index = first if first is not None else graph_intervention_config['benchmark_iterations']
    correction = trace.hard[index]
    base_valid = np.array_equal(base_context.H @ correction % 2, syndrome)
    logical_correct = graph_intervention_logical_label(base_context, correction) == truth
    success = base_valid and logical_correct
    probe_scores = transient_instability(context, trace, graph_intervention_config['probe_iterations'])
    after_I = local_instability(probe_scores, region)

    return {
        'syndrome_consistent': int(base_valid),
        'logical_correct': int(logical_correct),
        'repair': int(failure_type != 'not_failure' and success),
        'regression': int(failure_type == 'not_failure' and (not success)),
        'I_before': baseline_I,
        'I_after': after_I,
        'instability_reduction': baseline_I - after_I,
        'iterations': index,
        'runtime_seconds': runtime,
        'actual_edge_updates': int(index * context.H.sum()),
        'configured_edge_updates': int(graph_intervention_config['benchmark_iterations'] * context.H.sum()),
    }

damping_rows = []
scaling_rows = []
for graph_intervention_sample in tqdm(intervention_cases.itertuples(index=False), total=len(intervention_cases), desc='Damping and scaling', disable=graph_intervention_config['print_level'] == 1):
    graph_intervention_, graph_intervention_context = graph_intervention_context_for(2, graph_intervention_sample.p)
    graph_intervention_syndrome = graph_intervention_bit_array(graph_intervention_sample.syndrome)
    graph_intervention_baseline_trace = graph_intervention_trace_min_sum(graph_intervention_context, graph_intervention_syndrome, graph_intervention_config['benchmark_iterations'])
    baseline_scores = transient_instability(graph_intervention_context, graph_intervention_baseline_trace, graph_intervention_config['probe_iterations'])
    graph_intervention_region = graph_intervention_top_region(baseline_scores.composite, graph_intervention_config['primary_top_k'])
    baseline_I = local_instability(baseline_scores, graph_intervention_region)
    no_intervention = evaluate_modified(graph_intervention_context, graph_intervention_context, graph_intervention_syndrome, graph_intervention_syndrome, graph_intervention_sample.true_sector, graph_intervention_sample.failure_type, baseline_I, graph_intervention_region)
    damping_rows.append({
        'sample_id': graph_intervention_sample.sample_id,
        'failure_type': graph_intervention_sample.failure_type,
        'strategy': 'no_damping',
        'gamma_min': 0.0,
        'gamma_max': 0.0,
        'random_seed_index': -1,
        **no_intervention,
    })

    for gamma_min in graph_intervention_config['gamma_min_values']:
        for gamma_max in graph_intervention_config['gamma_max_values']:
            assert gamma_min < gamma_max < 1
            for mode in ['global', 'targeted']:
                graph_intervention_damping = graph_intervention_edge_maps(graph_intervention_context, baseline_scores.composite, gamma_min, gamma_max, mode)
                result = evaluate_modified(graph_intervention_context, graph_intervention_context, graph_intervention_syndrome, graph_intervention_syndrome, graph_intervention_sample.true_sector, graph_intervention_sample.failure_type, baseline_I, graph_intervention_region, damping=graph_intervention_damping)
                damping_rows.append({
                    'sample_id': graph_intervention_sample.sample_id,
                    'failure_type': graph_intervention_sample.failure_type,
                    'strategy': f'{mode}_damping',
                    'gamma_min': gamma_min,
                    'gamma_max': gamma_max,
                    'random_seed_index': -1,
                    'mean_damping': graph_intervention_damping.mean(),
                    **result,
                })

            for graph_intervention_seed_index in range(graph_intervention_config['targeting_random_seeds']):
                graph_intervention_rng = np.random.default_rng(graph_intervention_config['random_seed'] + int(graph_intervention_sample.sample_id) * 1009 + graph_intervention_seed_index)
                graph_intervention_damping = graph_intervention_edge_maps(graph_intervention_context, baseline_scores.composite, gamma_min, gamma_max, 'random', graph_intervention_rng)
                result = evaluate_modified(graph_intervention_context, graph_intervention_context, graph_intervention_syndrome, graph_intervention_syndrome, graph_intervention_sample.true_sector, graph_intervention_sample.failure_type, baseline_I, graph_intervention_region, damping=graph_intervention_damping)
                damping_rows.append({
                    'sample_id': graph_intervention_sample.sample_id,
                    'failure_type': graph_intervention_sample.failure_type,
                    'strategy': 'random_edge_damping',
                    'gamma_min': gamma_min,
                    'gamma_max': gamma_max,
                    'random_seed_index': graph_intervention_seed_index,
                    'mean_damping': graph_intervention_damping.mean(),
                    **result,
                })

    for alpha_min in graph_intervention_config['alpha_min_values']:
        alpha_high = graph_intervention_config['alpha_max']
        normalized = graph_intervention_percentile_rank(baseline_scores.composite)[graph_intervention_context.edge_variables]
        targeted_scaling = alpha_high - (alpha_high - alpha_min) * normalized
        scaling_maps = {
            'global_scaling': np.full(len(normalized), targeted_scaling.mean()),
            'targeted_scaling': targeted_scaling,
        }

        for graph_intervention_strategy, scaling in scaling_maps.items():
            result = evaluate_modified(graph_intervention_context, graph_intervention_context, graph_intervention_syndrome, graph_intervention_syndrome, graph_intervention_sample.true_sector, graph_intervention_sample.failure_type, baseline_I, graph_intervention_region, scaling=scaling)
            scaling_rows.append({
                'sample_id': graph_intervention_sample.sample_id,
                'failure_type': graph_intervention_sample.failure_type,
                'strategy': graph_intervention_strategy,
                'alpha_min': alpha_min,
                'alpha_max': alpha_high,
                'random_seed_index': -1,
                'mean_scaling': scaling.mean(),
                **result,
            })

        for graph_intervention_seed_index in range(graph_intervention_config['targeting_random_seeds']):
            graph_intervention_rng = np.random.default_rng(graph_intervention_config['random_seed'] + int(graph_intervention_sample.sample_id) * 2027 + graph_intervention_seed_index)
            scaling = graph_intervention_rng.permutation(targeted_scaling)
            result = evaluate_modified(graph_intervention_context, graph_intervention_context, graph_intervention_syndrome, graph_intervention_syndrome, graph_intervention_sample.true_sector, graph_intervention_sample.failure_type, baseline_I, graph_intervention_region, scaling=scaling)
            scaling_rows.append({
                'sample_id': graph_intervention_sample.sample_id,
                'failure_type': graph_intervention_sample.failure_type,
                'strategy': 'random_local_scaling',
                'alpha_min': alpha_min,
                'alpha_max': alpha_high,
                'random_seed_index': graph_intervention_seed_index,
                'mean_scaling': scaling.mean(),
                **result,
            })

damping_cases = pd.DataFrame(damping_rows)
scaling_cases = pd.DataFrame(scaling_rows)
damping_cases.to_csv(graph_intervention_trace_dir / 'damping_intervention_cases.csv', index=False)
scaling_cases.to_csv(graph_intervention_trace_dir / 'scaling_intervention_cases.csv', index=False)
damping_ablation = graph_intervention_conditional_outcome_rates(damping_cases.groupby(['strategy', 'gamma_min', 'gamma_max'], as_index=False).agg(
    evaluations=('sample_id', 'size'),
    unique_cases=('sample_id', 'nunique'),
    repair_count=('repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('failure_type', lambda x: np.sum(x != 'not_failure')),
    baseline_success_evaluations=('failure_type', lambda x: np.sum(x == 'not_failure')),
    syndrome_consistent_rate=('syndrome_consistent', 'mean'),
    mean_instability_reduction=('instability_reduction', 'mean'),
    mean_edge_updates=('actual_edge_updates', 'mean'),
    median_runtime_seconds=('runtime_seconds', 'median'),
))

damping_ablation.to_csv(graph_intervention_data_paths.output('damping_ablation_csv'), index=False)
scaling_ablation = graph_intervention_conditional_outcome_rates(scaling_cases.groupby(['strategy', 'alpha_min', 'alpha_max'], as_index=False).agg(
    evaluations=('sample_id', 'size'),
    unique_cases=('sample_id', 'nunique'),
    repair_count=('repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('failure_type', lambda x: np.sum(x != 'not_failure')),
    baseline_success_evaluations=('failure_type', lambda x: np.sum(x == 'not_failure')),
    syndrome_consistent_rate=('syndrome_consistent', 'mean'),
    mean_instability_reduction=('instability_reduction', 'mean'),
    mean_edge_updates=('actual_edge_updates', 'mean'),
    median_runtime_seconds=('runtime_seconds', 'median'),
))
scaling_ablation.to_csv(graph_intervention_data_paths.output('scaling_ablation_csv'), index=False)
damping_ablation.head()


## Experiment 7: unified intervention comparison

Place the intervention families in one comparison while retaining their repair and regression measurements. Success on an initially failing cohort alone is insufficient to rank a whole decoder.


In [ ]:
unified_rows = []
for graph_intervention_sample in tqdm(intervention_cases.itertuples(index=False), total=len(intervention_cases), desc='Unified interventions', disable=graph_intervention_config['print_level'] == 1):
    graph_intervention_sample_id = graph_intervention_sample.sample_id
    plain = damping_cases[(damping_cases.sample_id == graph_intervention_sample_id) & (damping_cases.strategy == 'no_damping')].iloc[0]
    targeted_row = targeting_cases[(targeting_cases.sample_id == graph_intervention_sample_id) & (targeting_cases.random_seed_index == 0)]
    selected_damping = damping_cases[(damping_cases.sample_id == graph_intervention_sample_id) & (damping_cases.gamma_min == graph_intervention_config['primary_gamma_min']) & (damping_cases.gamma_max == graph_intervention_config['primary_gamma_max'])]
    selected_scaling = scaling_cases[(scaling_cases.sample_id == graph_intervention_sample_id) & (scaling_cases.alpha_min == graph_intervention_config['primary_alpha_min'])]
    graph_intervention_source_rows = [('plain_bp', plain), ('random_redundant_row', targeted_row[targeted_row.strategy == 'fully_random'].iloc[0]), ('overlap_targeted_row', targeted_row[targeted_row.strategy == 'overlap_targeted'].iloc[0]), ('instability_reduction_targeted_row', targeted_row[targeted_row.strategy == 'instability_reduction_targeted'].iloc[0]), ('global_damping', selected_damping[selected_damping.strategy == 'global_damping'].iloc[0]), ('local_targeted_damping', selected_damping[selected_damping.strategy == 'targeted_damping'].iloc[0]), ('global_scaling', selected_scaling[selected_scaling.strategy == 'global_scaling'].iloc[0]), ('local_targeted_scaling', selected_scaling[selected_scaling.strategy == 'targeted_scaling'].iloc[0])]

    for graph_intervention_name, result in graph_intervention_source_rows:
        unified_rows.append({
            'sample_id': graph_intervention_sample_id,
            'failure_type': graph_intervention_sample.failure_type,
            'intervention': graph_intervention_name,
            'syndrome_consistent': int(result.syndrome_consistent),
            'logical_correct': int(result.logical_correct),
            'repair': int(result.repair),
            'regression': int(result.regression),
            'instability_reduction': float(result.get('instability_reduction', result.get('final_instability_reduction', np.nan))),
            'actual_edge_updates': int(result.get('actual_edge_updates', result.get('total_edge_updates'))),
            'runtime_seconds': float(result.runtime_seconds),
        })

    graph_intervention_, graph_intervention_context = graph_intervention_context_for(2, graph_intervention_sample.p)
    graph_intervention_syndrome = graph_intervention_bit_array(graph_intervention_sample.syndrome)
    graph_intervention_base_trace = graph_intervention_trace_min_sum(graph_intervention_context, graph_intervention_syndrome, graph_intervention_config['benchmark_iterations'])
    scores = transient_instability(graph_intervention_context, graph_intervention_base_trace, graph_intervention_config['probe_iterations'])
    graph_intervention_region = graph_intervention_top_region(scores.composite, graph_intervention_config['primary_top_k'])
    baseline_I = local_instability(scores, graph_intervention_region)
    graph_intervention_selected = primary_trials[primary_trials.sample_id == graph_intervention_sample_id].sort_values(['instability_gain', 'candidate_id'], ascending=[False, True]).iloc[0]
    graph_intervention_pool = {candidate.candidate_id: candidate for candidate in graph_intervention_generate_redundant_candidates(graph_intervention_context.H, graph_intervention_config['primary_max_row_weight'])}
    graph_intervention_candidate = graph_intervention_pool[int(graph_intervention_selected.candidate_id)]
    graph_intervention_augmented = graph_intervention_augmented_context(graph_intervention_context, graph_intervention_candidate.row)
    graph_intervention_augmented_s = graph_intervention_augmented_syndrome(graph_intervention_syndrome, graph_intervention_candidate.source_rows)
    graph_intervention_damping = graph_intervention_edge_maps(graph_intervention_augmented, scores.composite, graph_intervention_config['primary_gamma_min'], graph_intervention_config['primary_gamma_max'], 'targeted')
    graph_intervention_hybrid = evaluate_modified(graph_intervention_augmented, graph_intervention_context, graph_intervention_syndrome, graph_intervention_augmented_s, graph_intervention_sample.true_sector, graph_intervention_sample.failure_type, baseline_I, graph_intervention_region, damping=graph_intervention_damping)
    unified_rows.append({
        'sample_id': graph_intervention_sample_id,
        'failure_type': graph_intervention_sample.failure_type,
        'intervention': 'hybrid_row_plus_damping',
        **graph_intervention_hybrid,
    })

unified_cases = pd.DataFrame(unified_rows)
assert unified_cases.groupby('sample_id').size().eq(9).all()
unified_cases.to_csv(graph_intervention_trace_dir / 'unified_intervention_cases.csv', index=False)
unified_comparison = graph_intervention_conditional_outcome_rates(unified_cases.groupby('intervention', as_index=False).agg(
    cases=('sample_id', 'size'),
    repair_count=('repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('failure_type', lambda x: np.sum(x != 'not_failure')),
    baseline_success_evaluations=('failure_type', lambda x: np.sum(x == 'not_failure')),
    syndrome_consistent_rate=('syndrome_consistent', 'mean'),
    logical_correct_rate=('logical_correct', 'mean'),
    mean_instability_reduction=('instability_reduction', 'mean'),
    mean_edge_updates=('actual_edge_updates', 'mean'),
    median_runtime_seconds=('runtime_seconds', 'median'),
))

unified_comparison.to_csv(graph_intervention_data_paths.output('unified_intervention_comparison_csv'), index=False)
unified_comparison


## Experiments 8--9: iterative adaptation and failure-type effects

Apply repeated adaptation and break down outcomes by failure type. This examines whether useful actions remain available after an initial intervention and whether stable logical failures behave differently.


In [ ]:
iterative_rows = []
iterative_final_rows = []
maximum_rounds = max(graph_intervention_config['adaptation_rounds'])

for graph_intervention_sample in tqdm(intervention_cases.itertuples(index=False), total=len(intervention_cases), desc='Iterative graph adaptation', disable=graph_intervention_config['print_level'] == 1):
    graph_intervention_, graph_intervention_base_context = graph_intervention_context_for(2, graph_intervention_sample.p)
    graph_intervention_current = graph_intervention_base_context
    graph_intervention_base_syndrome = graph_intervention_bit_array(graph_intervention_sample.syndrome)
    graph_intervention_current_syndrome = graph_intervention_base_syndrome.copy()
    graph_intervention_cumulative_updates = 0
    stopped_reason = 'maximum_rounds'
    snapshots = [{
        'rows_added': 0,
        'instability': np.nan,
        'syndrome_consistent': int(graph_intervention_sample.benchmark_syndrome_consistent),
        'logical_correct': int(graph_intervention_sample.benchmark_logical_correct),
        'cumulative_edge_updates': 0,
        'stopping_reason': '',
    }]

    for graph_intervention_round_index in range(maximum_rounds):
        graph_intervention_probe = graph_intervention_trace_min_sum(graph_intervention_current, graph_intervention_current_syndrome, graph_intervention_config['probe_iterations'])
        scores = transient_instability(graph_intervention_current, graph_intervention_probe, graph_intervention_config['probe_iterations'])
        graph_intervention_region = graph_intervention_top_region(scores.composite, min(graph_intervention_config['primary_top_k'], graph_intervention_current.H.shape[1]))
        graph_intervention_current_I = local_instability(scores, graph_intervention_region)
        snapshots[-1]['instability'] = graph_intervention_current_I
        graph_intervention_candidates = graph_intervention_generate_redundant_candidates(graph_intervention_current.H, graph_intervention_config['primary_max_row_weight'])
        assert graph_intervention_candidates
        weighted = [graph_intervention_candidate_overlap(candidate, scores.composite, graph_intervention_region)[1] for candidate in graph_intervention_candidates]
        graph_intervention_rng = np.random.default_rng(graph_intervention_config['random_seed'] + int(graph_intervention_sample.sample_id) * 4099 + graph_intervention_round_index)
        graph_intervention_screened = balanced_screen(graph_intervention_candidates, weighted, graph_intervention_config['candidate_screen_limit'], graph_intervention_rng)
        graph_intervention_before = {value: local_instability(transient_instability(graph_intervention_current, graph_intervention_probe, value), graph_intervention_region) for value in graph_intervention_config['candidate_test_iterations']}
        graph_intervention_measured = []

        for graph_intervention_candidate in graph_intervention_screened:
            graph_intervention_primary = [row for row in candidate_probe(graph_intervention_current, graph_intervention_current_syndrome, graph_intervention_candidate, graph_intervention_region, graph_intervention_before) if row['test_iterations'] == graph_intervention_config['primary_candidate_test_iterations']][0]
            graph_intervention_measured.append((graph_intervention_primary['instability_gain'], graph_intervention_candidate, graph_intervention_primary))
            graph_intervention_cumulative_updates += graph_intervention_config['primary_candidate_test_iterations'] * int(graph_intervention_current.H.sum() + graph_intervention_candidate.row_weight)

        graph_intervention_gain, graph_intervention_selected, metadata = sorted(graph_intervention_measured, key=lambda item: (-item[0], item[1].candidate_id))[0]
        iterative_rows.append({
            'sample_id': graph_intervention_sample.sample_id,
            'failure_type': graph_intervention_sample.failure_type,
            'round': graph_intervention_round_index,
            'instability': graph_intervention_current_I,
            'selected_candidate_id': graph_intervention_selected.candidate_id,
            'selected_gain': graph_intervention_gain,
            'rows_added': graph_intervention_round_index,
            'cumulative_edge_updates': graph_intervention_cumulative_updates,
            'stopping_reason': '',
        })

        if graph_intervention_gain <= 0:
            stopped_reason = 'no_positive_gain'
            iterative_rows[-1]['stopping_reason'] = stopped_reason
            break
        graph_intervention_full = full_decode_candidate(graph_intervention_current, graph_intervention_current_syndrome, graph_intervention_sample.true_sector, graph_intervention_selected, graph_intervention_region)
        graph_intervention_cumulative_updates += graph_intervention_full['actual_edge_updates']
        graph_intervention_current_syndrome = graph_intervention_augmented_syndrome(graph_intervention_current_syndrome, graph_intervention_selected.source_rows)
        graph_intervention_current = graph_intervention_augmented_context(graph_intervention_current, graph_intervention_selected.row)
        post_probe = graph_intervention_trace_min_sum(graph_intervention_current, graph_intervention_current_syndrome, graph_intervention_config['probe_iterations'])
        post_scores = transient_instability(graph_intervention_current, post_probe, graph_intervention_config['probe_iterations'])
        post_region = graph_intervention_top_region(post_scores.composite, min(graph_intervention_config['primary_top_k'], graph_intervention_current.H.shape[1]))
        post_I = local_instability(post_scores, post_region)
        snapshots.append({
            'rows_added': graph_intervention_round_index + 1,
            'instability': post_I,
            'syndrome_consistent': graph_intervention_full['full_syndrome_consistent'],
            'logical_correct': graph_intervention_full['full_logical_correct'],
            'cumulative_edge_updates': graph_intervention_cumulative_updates,
            'stopping_reason': '',
        })

        if graph_intervention_full['full_syndrome_consistent']:
            stopped_reason = 'syndrome_consistent_solution'
            iterative_rows[-1]['stopping_reason'] = stopped_reason
            snapshots[-1]['stopping_reason'] = stopped_reason
            break

        if graph_intervention_full['final_normalized_late_residual'] < graph_intervention_config['stability_threshold']:
            stopped_reason = 'global_instability_below_threshold'
            iterative_rows[-1]['stopping_reason'] = stopped_reason
            snapshots[-1]['stopping_reason'] = stopped_reason
            break

    if not snapshots[-1]['stopping_reason']:
        snapshots[-1]['stopping_reason'] = stopped_reason
    if not (iterative_rows and iterative_rows[-1]['sample_id'] == graph_intervention_sample.sample_id and (iterative_rows[-1]['rows_added'] == snapshots[-1]['rows_added'])):
        iterative_rows.append({
            'sample_id': graph_intervention_sample.sample_id,
            'failure_type': graph_intervention_sample.failure_type,
            'round': snapshots[-1]['rows_added'],
            'instability': snapshots[-1]['instability'],
            'selected_candidate_id': np.nan,
            'selected_gain': np.nan,
            'rows_added': snapshots[-1]['rows_added'],
            'cumulative_edge_updates': snapshots[-1]['cumulative_edge_updates'],
            'stopping_reason': snapshots[-1]['stopping_reason'],
        })

    for limit in graph_intervention_config['adaptation_rounds']:
        available = [row for row in snapshots if row['rows_added'] <= limit]
        graph_intervention_last = available[-1]
        graph_intervention_reductions = [available[index + 1]['instability'] < available[index]['instability'] for index in range(len(available) - 1)]
        graph_intervention_valid = bool(graph_intervention_last['syndrome_consistent'])
        graph_intervention_correct = bool(graph_intervention_last['logical_correct'])
        graph_intervention_success = graph_intervention_valid and graph_intervention_correct
        iterative_final_rows.append({
            'sample_id': graph_intervention_sample.sample_id,
            'failure_type': graph_intervention_sample.failure_type,
            'max_rounds': limit,
            'rounds_executed': graph_intervention_last['rows_added'],
            'rows_added': graph_intervention_last['rows_added'],
            'monotonic_reduction_ratio': np.mean(graph_intervention_reductions) if graph_intervention_reductions else np.nan,
            'initial_instability': available[0]['instability'],
            'final_instability': available[-1]['instability'],
            'syndrome_consistent': int(graph_intervention_valid),
            'logical_correct': int(graph_intervention_correct),
            'repair': int(graph_intervention_sample.failure_type != 'not_failure' and graph_intervention_success),
            'regression': int(graph_intervention_sample.failure_type == 'not_failure' and (not graph_intervention_success)),
            'cumulative_edge_updates': graph_intervention_last['cumulative_edge_updates'],
            'stopping_reason': graph_intervention_last['stopping_reason'] or ('round_limit' if limit < snapshots[-1]['rows_added'] else stopped_reason),
        })

iterative_trajectories = pd.DataFrame(iterative_rows)
iterative_cases = pd.DataFrame(iterative_final_rows)
iterative_trajectories.to_csv(graph_intervention_trace_dir / 'iterative_graph_trajectories.csv', index=False)
iterative_cases.to_csv(graph_intervention_trace_dir / 'iterative_adaptation_cases.csv', index=False)
iterative_summary = graph_intervention_conditional_outcome_rates(iterative_cases.groupby('max_rounds', as_index=False).agg(
    cases=('sample_id', 'size'),
    repair_count=('repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('failure_type', lambda x: np.sum(x != 'not_failure')),
    baseline_success_evaluations=('failure_type', lambda x: np.sum(x == 'not_failure')),
    syndrome_consistent_rate=('syndrome_consistent', 'mean'),
    mean_rows_added=('rows_added', 'mean'),
    mean_monotonic_reduction_ratio=('monotonic_reduction_ratio', 'mean'),
    mean_instability_reduction=('initial_instability', lambda x: np.nan),
))

for graph_intervention_index, graph_intervention_row in iterative_summary.iterrows():
    graph_intervention_subset = iterative_cases[iterative_cases.max_rounds == graph_intervention_row.max_rounds]
    iterative_summary.loc[graph_intervention_index, 'mean_instability_reduction'] = (graph_intervention_subset.initial_instability - graph_intervention_subset.final_instability).mean()

iterative_summary.to_csv(graph_intervention_data_paths.output('iterative_adaptation_summary_csv'), index=False)
graph_intervention_failure_rows = []
for intervention, graph_intervention_group in unified_cases.groupby('intervention'):
    for graph_intervention_failure_type, graph_intervention_subset in graph_intervention_group.groupby('failure_type'):
        graph_intervention_failure_rows.append({
            'intervention': intervention,
            'failure_type': graph_intervention_failure_type,
            'cases': len(graph_intervention_subset),
            'repair_rate': graph_intervention_subset.repair.mean(),
            'regression_rate': graph_intervention_subset.regression.mean(),
            'syndrome_consistent_rate': graph_intervention_subset.syndrome_consistent.mean(),
            'mean_instability_reduction': graph_intervention_subset.instability_reduction.mean(),
        })

for limit, graph_intervention_group in iterative_cases.groupby('max_rounds'):
    for graph_intervention_failure_type, graph_intervention_subset in graph_intervention_group.groupby('failure_type'):
        graph_intervention_failure_rows.append({
            'intervention': f'iterative_q{limit}',
            'failure_type': graph_intervention_failure_type,
            'cases': len(graph_intervention_subset),
            'repair_rate': graph_intervention_subset.repair.mean(),
            'regression_rate': graph_intervention_subset.regression.mean(),
            'syndrome_consistent_rate': graph_intervention_subset.syndrome_consistent.mean(),
            'mean_instability_reduction': (graph_intervention_subset.initial_instability - graph_intervention_subset.final_instability).mean(),
        })

repair_by_failure_type = pd.DataFrame(graph_intervention_failure_rows)
repair_by_failure_type.to_csv(graph_intervention_data_paths.output('repair_by_failure_type_csv'), index=False)
iterative_summary


## Experiment 10: cross-p and cross-distance generalization

Repeat the configured comparisons across p and distance. Keep each population and candidate budget visible before drawing a generalization conclusion.


In [ ]:
def sampled_supports(
    num_variables,
    weight,
    limit,
    seed,
):
    total = int(np.math.comb(num_variables, weight)) if hasattr(np, 'math') else __import__('math').comb(num_variables, weight)
    rng = np.random.default_rng(seed)
    if total <= max(10000, limit):
        supports = list(combinations(range(num_variables), weight))
        rng.shuffle(supports)
        return (supports[:limit], total)
    result = set()

    while len(result) < limit:
        result.add(tuple(sorted(rng.choice(num_variables, weight, replace=False).tolist())))

    return (sorted(result), total)

p_values = sorted((float(value) for value in population.p.unique()))[:graph_intervention_config['cross_distance_p_limit']]
graph_intervention_cross_rows = []
cross_audit = []

for graph_intervention_level in tqdm(graph_intervention_config['cross_distance_levels'], desc='Cross-distance', disable=graph_intervention_config['print_level'] == 1):
    graph_intervention_distance, graph_intervention_reference = graph_intervention_context_for(graph_intervention_level, p_values[0])
    for error_weight in graph_intervention_config['cross_distance_error_weights']:
        supports, total = sampled_supports(graph_intervention_reference.H.shape[1], error_weight, graph_intervention_config['cross_distance_cases_per_stratum'] * 3, graph_intervention_config['random_seed'] + 100 * graph_intervention_level + error_weight)
        for p in p_values:
            graph_intervention_, graph_intervention_context = graph_intervention_context_for(graph_intervention_level, p)
            identities = {}
            for graph_intervention_support in supports:
                graph_intervention_error = np.zeros(graph_intervention_context.H.shape[1], dtype=np.uint8)
                graph_intervention_error[list(graph_intervention_support)] = 1
                graph_intervention_syndrome = graph_intervention_context.H @ graph_intervention_error % 2
                graph_intervention_truth = graph_intervention_logical_label(graph_intervention_context, graph_intervention_error)
                identities.setdefault((graph_intervention_bit_text(graph_intervention_syndrome), graph_intervention_truth), graph_intervention_error)

                if len(identities) >= graph_intervention_config['cross_distance_cases_per_stratum']:
                    break

            cross_audit.append({
                'distance': graph_intervention_distance,
                'p': p,
                'error_weight': error_weight,
                'physical_supports_total': total,
                'candidate_supports_sampled': len(supports),
                'deduplicated_inputs_selected': len(identities),
            })

            for graph_intervention_case_index, ((syndrome_text, graph_intervention_truth), graph_intervention_error) in enumerate(identities.items()):
                graph_intervention_syndrome = graph_intervention_bit_array(syndrome_text)
                graph_intervention_trace = graph_intervention_trace_min_sum(graph_intervention_context, graph_intervention_syndrome, graph_intervention_config['long_iterations'])
                graph_intervention_baseline = benchmark_result(graph_intervention_context, graph_intervention_syndrome, graph_intervention_truth, graph_intervention_error, graph_intervention_trace)
                scores = transient_instability(graph_intervention_context, graph_intervention_trace, graph_intervention_config['probe_iterations'])
                graph_intervention_region = graph_intervention_top_region(scores.composite, min(graph_intervention_config['primary_top_k'], graph_intervention_context.H.shape[1]))
                graph_intervention_before = {value: local_instability(transient_instability(graph_intervention_context, graph_intervention_trace, value), graph_intervention_region) for value in graph_intervention_config['candidate_test_iterations']}
                graph_intervention_candidates = graph_intervention_generate_redundant_candidates(graph_intervention_context.H, graph_intervention_config['primary_max_row_weight'])
                assert graph_intervention_candidates
                weighted = [graph_intervention_candidate_overlap(candidate, scores.composite, graph_intervention_region)[1] for candidate in graph_intervention_candidates]
                graph_intervention_rng = np.random.default_rng(graph_intervention_config['random_seed'] + graph_intervention_distance * 100000 + int(round(p * 1000000000.0)) + graph_intervention_case_index)
                graph_intervention_screened = balanced_screen(graph_intervention_candidates, weighted, min(graph_intervention_config['candidate_screen_limit'], len(graph_intervention_candidates)), graph_intervention_rng)
                graph_intervention_measured = []

                for graph_intervention_candidate in graph_intervention_screened:
                    metadata = [row for row in candidate_probe(graph_intervention_context, graph_intervention_syndrome, graph_intervention_candidate, graph_intervention_region, graph_intervention_before) if row['test_iterations'] == graph_intervention_config['primary_candidate_test_iterations']][0]
                    graph_intervention_measured.append((graph_intervention_candidate, metadata, graph_intervention_candidate_overlap(graph_intervention_candidate, scores.composite, graph_intervention_region)[1]))

                graph_intervention_gain_choice = sorted(graph_intervention_measured, key=lambda item: (-item[1]['instability_gain'], item[0].candidate_id))[0]
                graph_intervention_overlap_choice = sorted(graph_intervention_measured, key=lambda item: (-item[2], item[0].candidate_id))[0]
                random_choice = graph_intervention_measured[int(graph_intervention_rng.integers(len(graph_intervention_measured)))]
                probe_budget = sum((graph_intervention_config['primary_candidate_test_iterations'] * int(graph_intervention_context.H.sum() + item[0].row_weight) for item in graph_intervention_measured))

                for graph_intervention_strategy, choice in [('random', random_choice), ('overlap_targeted', graph_intervention_overlap_choice), ('instability_reduction_targeted', graph_intervention_gain_choice)]:
                    graph_intervention_candidate, metadata, graph_intervention_ = choice
                    graph_intervention_full = full_decode_candidate(graph_intervention_context, graph_intervention_syndrome, graph_intervention_truth, graph_intervention_candidate, graph_intervention_region)
                    graph_intervention_success = bool(graph_intervention_full['full_syndrome_consistent'] and graph_intervention_full['full_logical_correct'])
                    graph_intervention_baseline_success = graph_intervention_baseline['failure_type'] == 'not_failure'
                    graph_intervention_cross_rows.append({
                        'distance': graph_intervention_distance,
                        'p': p,
                        'error_weight': error_weight,
                        'case_index': graph_intervention_case_index,
                        'failure_type': graph_intervention_baseline['failure_type'],
                        'strategy': graph_intervention_strategy,
                        'instability_gain': metadata['instability_gain'],
                        'final_instability_reduction': metadata['I_before'] - graph_intervention_full['final_instability'],
                        'syndrome_consistent': graph_intervention_full['full_syndrome_consistent'],
                        'logical_correct': graph_intervention_full['full_logical_correct'],
                        'repair': int(not graph_intervention_baseline_success and graph_intervention_success),
                        'regression': int(graph_intervention_baseline_success and (not graph_intervention_success)),
                        'candidate_probe_edge_updates': probe_budget,
                        'total_edge_updates': probe_budget + graph_intervention_full['actual_edge_updates'],
                        'runtime_seconds': graph_intervention_full['runtime_seconds'],
                    })

cross_distance_cases = pd.DataFrame(graph_intervention_cross_rows)
cross_distance_cases.to_csv(graph_intervention_trace_dir / 'cross_distance_instability_targeting_cases.csv', index=False)
pd.DataFrame(cross_audit).to_csv(graph_intervention_data_paths.output('cross_distance_input_audit_csv'), index=False)
graph_intervention_cross_distance_summary = graph_intervention_conditional_outcome_rates(cross_distance_cases.groupby(['distance', 'strategy'], as_index=False).agg(
    evaluations=('case_index', 'size'),
    repair_count=('repair', 'sum'),
    regression_count=('regression', 'sum'),
    baseline_failure_evaluations=('failure_type', lambda x: np.sum(x != 'not_failure')),
    baseline_success_evaluations=('failure_type', lambda x: np.sum(x == 'not_failure')),
    syndrome_consistent_rate=('syndrome_consistent', 'mean'),
    mean_instability_gain=('instability_gain', 'mean'),
    mean_final_instability_reduction=('final_instability_reduction', 'mean'),
    mean_edge_updates=('total_edge_updates', 'mean'),
    median_runtime_seconds=('runtime_seconds', 'median'),
))

graph_intervention_cross_distance_summary.to_csv(graph_intervention_data_paths.output('cross_distance_instability_targeting_csv'), index=False)
graph_intervention_cross_distance_summary


## Validation and paired statistical analysis

In [ ]:
graph_intervention_, validation_context = graph_intervention_context_for(2, float(graph_intervention_population_cases.p.iloc[0]))
validation_candidate = graph_intervention_generate_redundant_candidates(validation_context.H, graph_intervention_config['primary_max_row_weight'])[0]
for graph_intervention_value in range(1 << min(12, validation_context.H.shape[1])):
    graph_intervention_correction = np.zeros(validation_context.H.shape[1], dtype=np.uint8)
    graph_intervention_correction[:min(12, validation_context.H.shape[1])] = graph_intervention_value >> np.arange(min(12, validation_context.H.shape[1])) & 1
    base_s = validation_context.H @ graph_intervention_correction % 2
    extra = int(validation_candidate.row @ graph_intervention_correction % 2)
    assert extra == int(base_s[validation_candidate.source_rows[0]] ^ base_s[validation_candidate.source_rows[1]])

validation_syndrome = graph_intervention_bit_array(graph_intervention_population_cases.iloc[0].syndrome)
plain_trace = graph_intervention_trace_min_sum(validation_context, validation_syndrome, graph_intervention_config['benchmark_iterations'])
zero_damping = graph_intervention_trace_min_sum(validation_context, validation_syndrome, graph_intervention_config['benchmark_iterations'], edge_damping=np.zeros(len(validation_context.edge_checks)))
unit_scaling = graph_intervention_trace_min_sum(validation_context, validation_syndrome, graph_intervention_config['benchmark_iterations'], edge_scaling=np.ones(len(validation_context.edge_checks)) * graph_intervention_config['ms_scaling_factor'])
np.testing.assert_allclose(plain_trace.c2v, zero_damping.c2v, atol=0, rtol=0)
np.testing.assert_allclose(plain_trace.c2v, unit_scaling.c2v, atol=0, rtol=0)
native_rows = []
validation_sample = graph_intervention_population_cases.sample(min(12, len(graph_intervention_population_cases)), random_state=graph_intervention_config['random_seed'])

for graph_intervention_row in validation_sample.itertuples(index=False):
    graph_intervention_, graph_intervention_context = graph_intervention_context_for(2, graph_intervention_row.p)
    graph_intervention_syndrome = graph_intervention_bit_array(graph_intervention_row.syndrome)
    graph_intervention_trace = graph_intervention_trace_min_sum(graph_intervention_context, graph_intervention_syndrome, graph_intervention_config['benchmark_iterations'])
    decoder = BpDecoder(
        graph_intervention_context.H,
        error_channel=graph_intervention_context.priors,
        max_iter=graph_intervention_config['benchmark_iterations'],
        bp_method='minimum_sum',
        ms_scaling_factor=graph_intervention_config['ms_scaling_factor'],
        schedule='parallel',
        input_vector_type='syndrome',
    )
    graph_intervention_native = np.asarray(decoder.decode(graph_intervention_syndrome), dtype=np.uint8)
    native_iteration = int(decoder.iter)
    graph_intervention_native_llr = np.asarray(decoder.log_prob_ratios)
    np.testing.assert_allclose(graph_intervention_trace.llr[native_iteration], graph_intervention_native_llr, rtol=graph_intervention_config['native_validation_tolerance'], atol=graph_intervention_config['native_validation_tolerance'])
    graph_intervention_hard_difference = graph_intervention_trace.hard[native_iteration] != graph_intervention_native

    if graph_intervention_hard_difference.any():
        assert np.max(np.abs(graph_intervention_trace.llr[native_iteration, graph_intervention_hard_difference])) <= graph_intervention_config['native_validation_tolerance']
        assert np.max(np.abs(graph_intervention_native_llr[graph_intervention_hard_difference])) <= graph_intervention_config['native_validation_tolerance']
    native_valid = np.array_equal(graph_intervention_context.H @ graph_intervention_native % 2, graph_intervention_syndrome)
    assert native_valid == bool(decoder.converge)
    native_rows.append({
        'sample_id': graph_intervention_row.sample_id,
        'hard_equal': int(not graph_intervention_hard_difference.any()),
        'instrumented_iteration': native_iteration,
        'native_iteration': native_iteration,
        'valid_equal': int(native_valid == graph_intervention_trace.syndrome_consistent[native_iteration]),
        'near_zero_hard_tie_count': int(graph_intervention_hard_difference.sum()),
        'max_abs_posterior_difference': float(np.max(np.abs(graph_intervention_trace.llr[native_iteration] - graph_intervention_native_llr))),
    })

native_validation = pd.DataFrame(native_rows)
native_validation.groupby(lambda _: True).agg(
    cases=('sample_id', 'size'),
    hard_equal=('hard_equal', 'sum'),
    valid_equal=('valid_equal', 'sum'),
    near_zero_hard_tie_count=('near_zero_hard_tie_count', 'sum'),
    max_abs_posterior_difference=('max_abs_posterior_difference', 'max'),
    maximum_iteration_difference=('native_iteration', lambda x: np.max(np.abs(x - native_validation.loc[x.index, 'instrumented_iteration']))),
).reset_index(drop=True).to_csv(graph_intervention_data_paths.output('graph_intervention_native_validation_csv'), index=False)
paired = targeting_cases.groupby(['sample_id', 'strategy'], as_index=False).agg(
    repair=('repair', 'mean'),
    regression=('regression', 'mean'),
    instability_gain=('instability_gain', 'mean'),
    total_edge_updates=('total_edge_updates', 'mean'),
)
pivot = paired.pivot(index='sample_id', columns='strategy')
comparison_rows = []
graph_intervention_rng = np.random.default_rng(graph_intervention_config['random_seed'])

for control in ['fully_random', 'weight_matched_random', 'overlap_targeted', 'anti_targeted']:
    for graph_intervention_metric in ['repair', 'regression', 'instability_gain']:
        graph_intervention_eligible = paired.sample_id
        if graph_intervention_metric == 'repair':
            graph_intervention_eligible = targeting_cases.loc[targeting_cases.failure_type != 'not_failure', 'sample_id'].unique()
        elif graph_intervention_metric == 'regression':
            graph_intervention_eligible = targeting_cases.loc[targeting_cases.failure_type == 'not_failure', 'sample_id'].unique()
        difference = (pivot.loc[pivot.index.intersection(graph_intervention_eligible), graph_intervention_metric]['instability_reduction_targeted'] - pivot.loc[pivot.index.intersection(graph_intervention_eligible), graph_intervention_metric][control]).dropna()
        graph_intervention_estimates = np.asarray([graph_intervention_rng.choice(difference, size=len(difference), replace=True).mean() for _ in tqdm(range(graph_intervention_config['bootstrap_repetitions']), desc=f'Bootstrap {control} {graph_intervention_metric}', leave=False, disable=graph_intervention_config['print_level'] == 1)])

        if np.allclose(difference, 0):
            statistic, p_value = (0.0, 1.0)
        else:
            statistic, p_value = wilcoxon(difference)
        comparison_rows.append({
            'comparison': f'instability_reduction_targeted_minus_{control}',
            'metric': graph_intervention_metric,
            'paired_cases': len(difference),
            'mean_difference': difference.mean(),
            'bootstrap_ci_low': np.quantile(graph_intervention_estimates, 0.025),
            'bootstrap_ci_high': np.quantile(graph_intervention_estimates, 0.975),
            'wilcoxon_statistic': statistic,
            'wilcoxon_p_value': p_value,
        })

paired_targeting = pd.DataFrame(comparison_rows)
paired_targeting.to_csv(graph_intervention_data_paths.output('targeting_paired_comparison_csv'), index=False)
repair_prediction = primary_trials[primary_trials.failure_type != 'not_failure'].copy()
model_rows = []

if repair_prediction.repair.nunique() == 2:
    graph_intervention_features = repair_prediction[['instability_gain']].to_numpy()
    graph_intervention_target = repair_prediction.repair.to_numpy()
    groups = repair_prediction.sample_id.to_numpy()
    folds = GroupKFold(n_splits=graph_intervention_config['cv_folds'])
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

    graph_intervention_score = cross_val_predict(model, graph_intervention_features, graph_intervention_target, groups=groups, cv=folds, method='predict_proba')[:, 1]
    model.fit(graph_intervention_features, graph_intervention_target)
    model_rows.append({
        'model': 'repair_from_short_probe_gain',
        'cases': len(graph_intervention_target),
        'unique_samples': repair_prediction.sample_id.nunique(),
        'group_cross_validated_auroc': roc_auc_score(graph_intervention_target, graph_intervention_score),
        'standardized_logistic_coefficient': model[-1].coef_[0, 0],
    })

pd.DataFrame(model_rows).to_csv(graph_intervention_data_paths.output('repair_gain_model_csv'), index=False)
paired_targeting


## Main figures

In [ ]:
map_samples = intervention_cases.groupby('failure_type', sort=True).head(8)
heatmap = variable_instability[variable_instability.sample_id.isin(map_samples.sample_id)].pivot(index='sample_id', columns='variable_id', values='trajectory_instability')
graph_intervention_fig, graph_intervention_axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
sns.heatmap(heatmap, ax=graph_intervention_axes[0], cmap='magma')
graph_intervention_axes[0].set_title('Transient instability map')
trajectory_distribution = graph_intervention_population_cases.melt(
    id_vars='failure_type',
    value_vars=['max_trajectory_instability', 'mean_trajectory_instability'],
    var_name='summary',
    value_name='trajectory_instability',
)

sns.boxplot(
    data=trajectory_distribution,
    x='failure_type',
    y='trajectory_instability',
    hue='summary',
    ax=graph_intervention_axes[1],
)
graph_intervention_axes[1].tick_params(axis='x', rotation=30)
graph_intervention_axes[1].set_yscale('log')

plt.show()
representatives = intervention_cases.groupby('failure_type', sort=True).head(1)
graph_intervention_fig, graph_intervention_axes = plt.subplots(len(representatives), 1, figsize=(12, 3 * len(representatives)), constrained_layout=True)
graph_intervention_axes = np.atleast_1d(graph_intervention_axes)

for axis, graph_intervention_row in zip(graph_intervention_axes, representatives.itertuples(index=False)):
    data = iteration_instability[iteration_instability.sample_id == graph_intervention_row.sample_id]
    sns.lineplot(
        data=data,
        x='iteration',
        y='normalized_message_residual',
        hue='variable_id',
        legend=False,
        ax=axis,
    )
    axis.set_title(graph_intervention_row.failure_type)
    axis.set_yscale('log')

plt.show()
graph_intervention_fig, graph_intervention_axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
sns.barplot(
    data=overlap_summary,
    x='hops',
    y='overlap_mean',
    hue='outcome_class',
    ax=graph_intervention_axes[0],
)
sns.barplot(
    data=overlap_summary,
    x='hops',
    y='enrichment_mean',
    hue='outcome_class',
    ax=graph_intervention_axes[1],
)
graph_intervention_axes[0].set_title('Instability-residual overlap')
graph_intervention_axes[1].set_title('Random-support enrichment')

plt.show()
overlay_samples = overlap_input.groupby('stable_outcome', sort=True).head(1)
graph_intervention_fig, graph_intervention_axes = plt.subplots(1, len(overlay_samples), figsize=(8 * len(overlay_samples), 6), constrained_layout=True)
graph_intervention_axes = np.atleast_1d(graph_intervention_axes)

for axis, graph_intervention_row in zip(graph_intervention_axes, overlay_samples.itertuples(index=False)):
    graph_intervention_, graph_intervention_context = graph_intervention_context_for(2, graph_intervention_row.p)
    graph = nx.Graph()
    variable_nodes = [f'v{i}' for i in range(graph_intervention_context.H.shape[1])]
    check_nodes = [f'c{i}' for i in range(graph_intervention_context.H.shape[0])]
    graph.add_nodes_from(variable_nodes, bipartite=0)
    graph.add_nodes_from(check_nodes, bipartite=1)
    graph.add_edges_from(((f'c{c}', f'v{v}') for c, v in zip(graph_intervention_context.edge_checks, graph_intervention_context.edge_variables)))
    graph_intervention_position = nx.bipartite_layout(graph, variable_nodes, align='horizontal')
    values = variable_instability[variable_instability.sample_id == graph_intervention_row.sample_id].sort_values('variable_id').composite_instability
    residual = set(np.flatnonzero(graph_intervention_bit_array(graph_intervention_row.residual)))
    nx.draw_networkx_edges(graph, graph_intervention_position, alpha=0.25, ax=axis)
    nx.draw_networkx_nodes(
        graph,
        graph_intervention_position,
        nodelist=check_nodes,
        node_shape='s',
        node_color='lightgray',
        node_size=80,
        ax=axis,
    )
    nx.draw_networkx_nodes(
        graph,
        graph_intervention_position,
        nodelist=variable_nodes,
        node_color=values,
        cmap='magma',
        node_size=120,
        edgecolors=['red' if i in residual else 'black' for i in range(len(variable_nodes))],
        linewidths=2,
        ax=axis,
    )
    axis.set_title(f'{graph_intervention_row.stable_outcome}: red outline = residual support')
    axis.set_axis_off()

plt.show()
graph_intervention_fig, graph_intervention_axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
sns.boxplot(
    data=primary_trials,
    x='repair',
    y='instability_gain',
    ax=graph_intervention_axes[0],
)
sns.barplot(
    data=repair_vs_gain,
    x='gain_percentile_bin',
    y='repair_fraction',
    ax=graph_intervention_axes[1],
)
graph_intervention_axes[0].set_title('Short-probe gain vs repair')
graph_intervention_axes[1].set_title('Repair by gain percentile')

plt.show()
graph_intervention_fig, graph_intervention_axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
sns.barplot(
    data=targeting_ablation,
    x='strategy',
    y='repair_rate',
    ax=graph_intervention_axes[0],
)
sns.scatterplot(
    data=targeting_ablation,
    x='mean_edge_updates',
    y='repair_rate',
    hue='strategy',
    s=100,
    ax=graph_intervention_axes[1],
)
graph_intervention_axes[0].tick_params(axis='x', rotation=30)
graph_intervention_axes[0].set_title('Targeting ablation')
graph_intervention_axes[1].set_title('Repair vs actual edge-update budget')

plt.show()
graph_intervention_fig, graph_intervention_axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
sns.scatterplot(
    data=unified_comparison,
    x='mean_instability_reduction',
    y='repair_rate',
    hue='intervention',
    s=100,
    ax=graph_intervention_axes[0],
)
sns.lineplot(
    data=iterative_trajectories,
    x='round',
    y='instability',
    hue='failure_type',
    estimator='median',
    ax=graph_intervention_axes[1],
)
sns.barplot(
    data=graph_intervention_cross_distance_summary,
    x='distance',
    y='repair_rate',
    hue='strategy',
    ax=graph_intervention_axes[2],
)
graph_intervention_axes[0].set_title('Unified intervention')
graph_intervention_axes[1].set_title('Iterative adaptation')
graph_intervention_axes[2].set_title('Cross-distance targeting')

plt.show()


## Interpretation guardrails

- Success means syndrome validity **and** equality with the physical error's true logical sector; exact physical-support equality is not required.
- Candidate selection uses only probe dynamics. Full logical outcomes are evaluation-only.
- Candidate screening is balanced across high-overlap, low-overlap, and random rows so that overlap and anti-targeted controls remain identifiable.
- Candidate-probe edge updates are charged to every targeting strategy, even if a control ignores the measured gains.
- Cross-distance results use bounded fixed-seed samples and are not population LER estimates.
- Reduced instability without improved logical correctness supports a dynamics effect, not logical-sector resolution.